# Phase 5: GitHub Repo Analyzer with Real-Time Event & Token Streaming

This notebook implements **Phase 5** from `Sprinter_Implementation_Plan.md`.

### Core Architectural Features:
1. **Complete Isolation**: Developed and validated independently from the QA agent before integration into the Phase 6 DocGen and Phase 7 Supervisor pipelines.
2. **GitHub MCP Server Integration**: Connects to `@modelcontextprotocol/server-github` via `langchain-mcp-adapters` for authenticated repo tree traversal and file content reading.
3. **Map-Reduce Fan-Out Architecture**: Utilizes LangGraph's dynamic `Send()` API and `operator.add` reducer to analyze multiple files concurrently.
4. **Subgraph-Inside-Node Pattern (Isolated State)**: Encapsulated behind an adapter node so parent graphs (DocGen, Supervisor) only pass `(repo_url, query)` and receive `repo_summary`, avoiding state pollution and reducer leakage.
5. **Real-Time Event & Token Streaming**: Full visibility into workflow execution events—node lifecycles, file discovery, parallel worker dispatch, and token-by-token synthesis streaming.


## 1. Imports and Dependencies
Import standard libraries, Pydantic for validation, LangChain / Mistral for LLM execution, LangGraph for state orchestration, and `langchain-mcp-adapters` for GitHub MCP integration.


In [ ]:
import os
import sys
import json
import time
import asyncio
import operator
from typing import TypedDict, Annotated, List, Dict, Any, Optional, Literal, Tuple
from urllib.parse import urlparse
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# Ensure UTF-8 output encoding for console/terminals on Windows
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Ensure workspace root is in python path
sys.path.insert(0, os.getcwd())

# LangChain & LangGraph
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mistralai import ChatMistralAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langsmith import traceable

# MCP Adapters
from langchain_mcp_adapters.client import MultiServerMCPClient

print("All core dependencies imported successfully.")


## 2. Configuration & Environment Setup
Loads environment variables (`GITHUB_ACCESS_TOKEN`, `MISTRAL_API_KEY`, `LANGSMITH_*`) with robust path and key cleaning.


In [ ]:
class Config:
    """Central configuration management for the GitHub Repo Analyzer."""
    
    def __init__(self):
        # Locate .env file reliably in current or parent directory
        env_path = os.path.join(os.getcwd(), ".env")
        if os.path.exists(env_path):
            load_dotenv(dotenv_path=env_path, override=True)
        else:
            load_dotenv(override=True)
            
        raw_token = os.getenv("GITHUB_ACCESS_TOKEN") or os.getenv("GITHUB_TOKEN") or ""
        self.github_token = raw_token.strip().strip('"').strip("'")
        
        raw_key = os.getenv("MISTRAL_API_KEY") or ""
        self.mistral_api_key = raw_key.strip().strip('"').strip("'")
        
        self.langsmith_tracing = os.getenv("LANGSMITH_TRACING", "false").lower() == "true"
        self.langsmith_project = os.getenv("LANGSMITH_PROJECT", "sprinter-phase5-repo-analyzer")
        
        if not self.github_token:
            print("Warning: GITHUB_ACCESS_TOKEN not found. API rate limits will be restricted.")
        else:
            print("GitHub Access Token detected.")
            
        if not self.mistral_api_key:
            raise ValueError("MISTRAL_API_KEY is required in .env.")
        else:
            print("Mistral API Key detected.")

config = Config()


## 3. Pydantic Models for Structured Output
- `FileSelection`: Enforces structured file prioritization with reasoning.
- `FileSummary`: Enforces structured output per file from parallel workers.


In [ ]:
class FileSelection(BaseModel):
    """Structured output for files selected to analyze."""
    files_to_analyze: List[str] = Field(
        ...,
        description="List of 3 to 8 relevant file paths from the repository tree that are critical to answering the user query."
    )
    reasoning: str = Field(
        ...,
        description="Justification explaining why these specific files were selected for architectural analysis."
    )


class FileSummary(BaseModel):
    """Structured summary of an individual file analyzed by a parallel worker."""
    file_path: str = Field(..., description="The repository file path.")
    summary: str = Field(..., description="Detailed summary of the file's role, classes, endpoints, or core logic.")
    key_symbols: List[str] = Field(
        default_factory=list,
        description="Key classes, functions, or configurations defined in this file."
    )
    relevance_to_query: Literal["high", "medium", "low"] = Field(
        ...,
        description="Degree of relevance to the user's specific query."
    )


## 4. State Definitions (`RepoAnalysisState` & `WorkerState`)
- `RepoAnalysisState`: Internal state of the repo analyzer subgraph. Uses `file_summaries: Annotated[List[dict], operator.add]` as a reducer.
- `WorkerState`: Minimal state passed to each parallel worker via `Send()`.


In [ ]:
class RepoAnalysisState(TypedDict):
    """Internal State schema for the GitHub Repo Analyzer Subgraph."""
    repo_url: str
    user_query: str
    owner: str
    repo: str
    branch: Optional[str]
    file_tree: List[str]
    selected_files: List[str]
    selection_reasoning: str
    file_summaries: Annotated[List[dict], operator.add]  # Dynamic Map-Reduce Reducer
    combined_summary: str
    error: Optional[str]


class WorkerState(TypedDict):
    """State passed to each parallel worker via LangGraph Send()."""
    file_path: str
    user_query: str
    owner: str
    repo: str
    branch: Optional[str]


## 5. GitHub MCP Client Wrapper
Connects to `@modelcontextprotocol/server-github` over stdio via `langchain-mcp-adapters`.
Provides:
- URL parsing (`owner`, `repo`, `branch`)
- Asynchronous tree traversal with concurrency control and ignore filters
- Truncation-safe file content retrieval


In [ ]:
DEFAULT_IGNORED_DIRS = {
    ".git", "node_modules", "__pycache__", ".pytest_cache", ".venv", "venv",
    "env", ".env", "dist", "build", ".idea", ".vscode", ".egg-info", "target",
    "bin", "obj", ".mypy_cache", ".ruff_cache", "coverage", ".next", ".nuxt"
}

DEFAULT_IGNORED_EXTS = {
    ".png", ".jpg", ".jpeg", ".gif", ".ico", ".svg", ".webp", ".mp4", ".mp3",
    ".zip", ".tar", ".gz", ".7z", ".rar", ".exe", ".dll", ".so", ".dylib",
    ".pyc", ".pyo", ".pyd", ".pkl", ".joblib", ".bin", ".weights", ".h5", ".onnx",
    ".pdf", ".docx", ".xlsx", ".csv", ".tsv", ".sqlite", ".db", ".lock"
}

MAX_FILE_SIZE_BYTES = 80 * 1024  # 80 KB limit per file fetch


def parse_github_url(url: str) -> Tuple[str, str, Optional[str]]:
    """Extract owner, repo, and optional branch from GitHub URL."""
    cleaned = url.strip()
    if cleaned.endswith(".git"):
        cleaned = cleaned[:-4]
        
    if not cleaned.startswith("http://") and not cleaned.startswith("https://"):
        parts = [p for p in cleaned.split("/") if p]
        if len(parts) >= 2:
            return parts[0], parts[1], None
        raise ValueError(f"Invalid GitHub identifier: '{url}'. Expected 'owner/repo'.")

    parsed = urlparse(cleaned)
    path_parts = [p for p in parsed.path.strip("/").split("/") if p]
    if len(path_parts) < 2:
        raise ValueError(f"Invalid GitHub URL: '{url}'. Missing owner and repository.")

    owner, repo = path_parts[0], path_parts[1]
    branch = path_parts[3] if len(path_parts) >= 4 and path_parts[2] == "tree" else None
    return owner, repo, branch


class GitHubMCPClient:
    """Encapsulates GitHub MCP tool calls via langchain-mcp-adapters."""

    def __init__(self, token: Optional[str] = None):
        self.token = token or config.github_token
        self._client: Optional[MultiServerMCPClient] = None
        self._tools_cache: Optional[Dict[str, Any]] = None
        self._lock = asyncio.Lock()

    def _get_npx_command(self) -> str:
        return "npx.cmd" if sys.platform == "win32" else "npx"

    async def get_client(self) -> MultiServerMCPClient:
        if self._client is None:
            async with self._lock:
                if self._client is None:
                    env_vars = {}
                    if self.token:
                        env_vars["GITHUB_PERSONAL_ACCESS_TOKEN"] = self.token
                    
                    self._client = MultiServerMCPClient({
                        "github": {
                            "transport": "stdio",
                            "command": self._get_npx_command(),
                            "args": ["-y", "@modelcontextprotocol/server-github"],
                            "env": env_vars
                        }
                    })
        return self._client

    async def get_tool(self, tool_name: str) -> Any:
        if self._tools_cache is None:
            async with self._lock:
                if self._tools_cache is None:
                    client = await self.get_client()
                    tools_list = await client.get_tools()
                    self._tools_cache = {t.name: t for t in tools_list}
        return self._tools_cache.get(tool_name)

    async def fetch_repo_tree(
        self,
        owner: str,
        repo: str,
        branch: Optional[str] = None,
        max_depth: int = 3,
        max_files: int = 150
    ) -> List[str]:
        """Recursively list repo files filtering non-code/lock files."""
        tool = await self.get_tool("get_file_contents")
        if not tool:
            raise RuntimeError("MCP tool 'get_file_contents' not found on GitHub MCP server.")

        collected_files: List[str] = []
        semaphore = asyncio.Semaphore(5)

        async def _traverse(path: str = "", depth: int = 0):
            if depth > max_depth or len(collected_files) >= max_files:
                return

            args = {"owner": owner, "repo": repo, "path": path}
            if branch:
                args["branch"] = branch

            async with semaphore:
                try:
                    res = await tool.ainvoke(args)
                except Exception:
                    return

            text = res[0]["text"] if isinstance(res, list) and res and "text" in res[0] else str(res)
            try:
                entries = json.loads(text)
            except Exception:
                return

            if not isinstance(entries, list):
                return

            sub_tasks = []
            for entry in entries:
                if len(collected_files) >= max_files:
                    break
                t, p, name = entry.get("type"), entry.get("path", ""), entry.get("name", "")
                if t == "file":
                    ext = os.path.splitext(name)[1].lower()
                    if ext not in DEFAULT_IGNORED_EXTS and name not in DEFAULT_IGNORED_DIRS:
                        collected_files.append(p)
                elif t == "dir":
                    if name not in DEFAULT_IGNORED_DIRS and not name.startswith("."):
                        sub_tasks.append(_traverse(p, depth + 1))

            if sub_tasks:
                await asyncio.gather(*sub_tasks)

        await _traverse("", depth=0)
        return sorted(collected_files)

    async def fetch_file_content(
        self,
        owner: str,
        repo: str,
        file_path: str,
        branch: Optional[str] = None
    ) -> Dict[str, Any]:
        """Fetch raw file content via MCP get_file_contents."""
        tool = await self.get_tool("get_file_contents")
        if not tool:
            raise RuntimeError("MCP tool 'get_file_contents' not found.")

        args = {"owner": owner, "repo": repo, "path": file_path}
        if branch:
            args["branch"] = branch

        try:
            res = await tool.ainvoke(args)
        except Exception as e:
            return {"file_path": file_path, "content": f"[Error reading file: {e}]", "truncated": False}

        text = res[0]["text"] if isinstance(res, list) and res and "text" in res[0] else str(res)
        try:
            data = json.loads(text)
        except Exception:
            return {"file_path": file_path, "content": text[:MAX_FILE_SIZE_BYTES], "truncated": len(text) > MAX_FILE_SIZE_BYTES}

        if isinstance(data, dict):
            raw = data.get("content", "")
            truncated = len(raw) > MAX_FILE_SIZE_BYTES
            if truncated:
                raw = raw[:MAX_FILE_SIZE_BYTES] + "\n\n... [Truncated due to size limit] ..."
            return {"file_path": file_path, "content": raw, "truncated": truncated}

        return {"file_path": file_path, "content": str(data), "truncated": False}

github_mcp = GitHubMCPClient()
print("GitHub MCP Client configured.")


## 6. LLM Client Wrapper & Structured Output Handler
Wraps `ChatMistralAI` with `mistral-medium-2505` and self-repair validation loops for guaranteed structured outputs.


In [ ]:
class LLMClient:
    """Wrapper for ChatMistralAI with configuration and retries."""

    def __init__(
        self,
        model: str = "mistral-medium-2505",
        temperature: float = 0.1,
        max_retries: int = 3,
        timeout: int = 60,
    ):
        self.model_name = model
        self.temperature = temperature
        self.llm = ChatMistralAI(
            model=model,
            temperature=temperature,
            max_retries=max_retries,
            timeout=timeout,
            mistral_api_key=config.mistral_api_key
        )

    def with_structured_output(self, schema: type[BaseModel]):
        return self.llm.with_structured_output(schema)


class StructuredOutputNode:
    """Executes structured output with a 1-attempt repair loop."""

    def __init__(self, llm_client: LLMClient, schema: type[BaseModel], system_prompt: str):
        self.llm_client = llm_client
        self.schema = schema
        self.system_prompt = system_prompt
        self.structured_llm = llm_client.with_structured_output(schema)

    async def invoke_async(self, user_content: str) -> BaseModel:
        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=user_content)
        ]
        try:
            return await self.structured_llm.ainvoke(messages)
        except Exception as e:
            # Repair loop: feed validation error back to LLM for correction
            repair_messages = [
                SystemMessage(content=self.system_prompt),
                HumanMessage(content=user_content),
                HumanMessage(content=f"Your previous response caused validation error: {str(e)}. Please correct and output valid JSON.")
            ]
            return await self.structured_llm.ainvoke(repair_messages)

llm_client = LLMClient(model="mistral-medium-2505")
print("LLM Client and StructuredOutputNode ready.")


## 7. Graph Nodes with Real-Time Event Instrumentation
Each node is instrumented with event printing so you can see:
1. When nodes start and complete.
2. File tree discovery statistics.
3. Selected files and reasoning.
4. Parallel worker execution and summary extraction.
5. Live token streaming during the final synthesis.


In [ ]:
# Node 1: Fetch Repo Tree
@traceable(name="fetch_repo_tree_node")
async def fetch_repo_tree_node(state: RepoAnalysisState) -> dict:
    """Extracts repo tree using GitHub MCP."""
    print(f"\n📁 [NODE: fetch_repo_tree] Scanning repository tree via GitHub MCP...")
    try:
        owner, repo, branch = parse_github_url(state["repo_url"])
        print(f"   Target: {owner}/{repo} (Branch: {branch or 'default'})")
    except Exception as e:
        err_msg = f"Invalid URL: {str(e)}"
        print(f"   ❌ {err_msg}")
        return {"error": err_msg, "file_tree": [], "owner": "", "repo": "", "branch": None}

    file_tree = await github_mcp.fetch_repo_tree(owner, repo, branch)
    print(f"   ✅ Discovered {len(file_tree)} eligible code files.")
    if file_tree:
        preview_sample = file_tree[:min(5, len(file_tree))]
        print(f"   Preview (first {len(preview_sample)}): {preview_sample}")

    return {
        "owner": owner,
        "repo": repo,
        "branch": branch,
        "file_tree": file_tree,
        "error": None if file_tree else "No files found in repository."
    }


# Node 2: Select Relevant Files
FILE_SELECTION_PROMPT = """You are an expert software architect analyzing a code repository.
Given a user query and the list of files in the repository:
1. Identify 3 to 8 files most relevant and critical to addressing the query.
2. Prioritize entry points, routing files, service definitions, or core modules.
3. Every file path you select MUST match EXACTLY one of the paths in the provided file list. Do NOT invent paths."""

file_selector = StructuredOutputNode(
    llm_client=llm_client,
    schema=FileSelection,
    system_prompt=FILE_SELECTION_PROMPT
)

@traceable(name="select_relevant_files_node")
async def select_relevant_files_node(state: RepoAnalysisState) -> dict:
    """Selects critical files based on user query and file tree."""
    print(f"\n🎯 [NODE: select_relevant_files] Analyzing repository structure against user query...")
    tree = state.get("file_tree", [])
    if not tree:
        print("   ⚠️ No file tree available to select from.")
        return {"selected_files": [], "selection_reasoning": "No file tree available."}

    tree_preview = "\n".join(tree[:120])
    prompt_content = f"""User Query: {state['user_query']}

Repository File Tree ({len(tree)} files):
{tree_preview}

Select 3 to 8 of the most relevant files to analyze this query."""

    try:
        selection: FileSelection = await file_selector.invoke_async(prompt_content)
        # Validate that selected files actually exist in tree
        valid_files = [f for f in selection.files_to_analyze if f in tree]
        if not valid_files:
            valid_files = tree[:min(5, len(tree))]
            print("   ⚠️ Validation fallback: using top files from tree.")

        print(f"   ✅ Selected {len(valid_files)} files: {valid_files}")
        print(f"   💡 Selection Reasoning: {selection.reasoning}")
        return {
            "selected_files": valid_files,
            "selection_reasoning": selection.reasoning
        }
    except Exception as e:
        fallback_files = tree[:min(4, len(tree))]
        print(f"   ⚠️ Selection error, using fallback: {fallback_files}")
        return {
            "selected_files": fallback_files,
            "selection_reasoning": f"Fallback selection due to error: {str(e)}"
        }


# Conditional Edge: Dynamic Fan-Out using LangGraph Send()
def distribute_files(state: RepoAnalysisState) -> List[Send]:
    """Generates parallel worker tasks via LangGraph Send() API."""
    selected = state.get("selected_files", [])
    print(f"\n⚡ [FAN-OUT: Send()] Launching {len(selected)} parallel worker tasks...")
    tasks = []
    for path in selected:
        print(f"   -> Enqueued worker for file: '{path}'")
        tasks.append(Send(
            "summarize_file",
            WorkerState(
                file_path=path,
                user_query=state["user_query"],
                owner=state["owner"],
                repo=state["repo"],
                branch=state.get("branch")
            )
        ))
    return tasks


# Node 3 (Worker): Summarize Individual File
FILE_SUMMARY_PROMPT = """You are a code analysis specialist.
Analyze the provided code file in relation to the user query:
1. Summarize the role of this file in the application.
2. Note key classes, functions, or configurations.
3. Assess relevance ('high', 'medium', or 'low')."""

file_summarizer = StructuredOutputNode(
    llm_client=llm_client,
    schema=FileSummary,
    system_prompt=FILE_SUMMARY_PROMPT
)

@traceable(name="summarize_file_worker")
async def summarize_file_node(state: WorkerState) -> dict:
    """Parallel worker fetching file content via MCP and summarizing it."""
    file_path = state["file_path"]
    print(f"  🔍 [WORKER START] Fetching content for: '{file_path}'...")
    
    file_info = await github_mcp.fetch_file_content(
        owner=state["owner"],
        repo=state["repo"],
        file_path=file_path,
        branch=state.get("branch")
    )

    user_prompt = f"""User Query: {state['user_query']}
File Path: {file_path}

File Content:
```
{file_info['content']}
```

Summarize this file's implementation details and key symbols related to the query."""

    try:
        summary: FileSummary = await file_summarizer.invoke_async(user_prompt)
        print(f"  ✅ [WORKER DONE] Summarized '{file_path}' (Relevance: {summary.relevance_to_query.upper()}, Symbols: {len(summary.key_symbols)})")
        return {
            "file_summaries": [summary.model_dump()]
        }
    except Exception as e:
        print(f"  ❌ [WORKER ERROR] Error in '{file_path}': {str(e)}")
        return {
            "file_summaries": [{
                "file_path": file_path,
                "summary": f"Could not summarize file: {str(e)}",
                "key_symbols": [],
                "relevance_to_query": "low"
            }]
        }


# Node 4: Reduce Summaries (With Live Token Streaming)
@traceable(name="reduce_summaries_node")
async def reduce_summaries_node(state: RepoAnalysisState) -> dict:
    """Synthesizes individual file summaries into an architectural answer with live token streaming."""
    summaries = state.get("file_summaries", [])
    print(f"\n🧠 [NODE: reduce_summaries] Synthesizing {len(summaries)} file analyses into final architectural document...")
    if not summaries:
        return {"combined_summary": "No file summaries could be generated."}

    formatted_summaries = []
    for s in summaries:
        formatted_summaries.append(
            f"### `{s['file_path']}` (Relevance: {s.get('relevance_to_query', 'N/A')})\n"
            f"- **Symbols:** {', '.join(s.get('key_symbols', [])) if s.get('key_symbols') else 'None listed'}\n"
            f"- **Summary:** {s['summary']}\n"
        )

    all_analyses = "".join(formatted_summaries)
    synthesis_prompt = f"""You are an expert technical documentation architect.
Synthesize the analyzed files to answer the user query comprehensively.

User Query: {state['user_query']}
Repository: {state['owner']}/{state['repo']}
Selection Reasoning: {state.get('selection_reasoning', '')}

File Analyses:
{all_analyses}

Generate a structured architectural overview explaining how the repository addresses the user query.
Use markdown headings, bullet points, and code references."""

    print("\n--- [LIVE TOKEN STREAMING OUTPUT] ---\n")
    full_text = ""
    async for chunk in llm_client.llm.astream([
        SystemMessage(content="You are an expert software documentation architect."),
        HumanMessage(content=synthesis_prompt)
    ]):
        delta = chunk.content
        if delta:
            full_text += delta
            print(delta, end="", flush=True)

    print("\n\n--- [END OF STREAMING] ---\n")
    return {"combined_summary": full_text}

print("All instrumented graph nodes defined.")


## 8. Graph Construction & Compilation (`RepoAnalyzerGraph`)
Assembles the Map-Reduce flow:
`START -> fetch_repo_tree -> select_relevant_files -> (Send fan-out) -> summarize_file -> reduce_summaries -> END`


In [ ]:
def build_repo_analyzer_graph():
    """Builds and compiles the Phase 5 standalone Map-Reduce Subgraph."""
    builder = StateGraph(RepoAnalysisState)

    builder.add_node("fetch_repo_tree", fetch_repo_tree_node)
    builder.add_node("select_relevant_files", select_relevant_files_node)
    builder.add_node("summarize_file", summarize_file_node)
    builder.add_node("reduce_summaries", reduce_summaries_node)

    builder.add_edge(START, "fetch_repo_tree")
    builder.add_edge("fetch_repo_tree", "select_relevant_files")
    builder.add_conditional_edges("select_relevant_files", distribute_files, ["summarize_file"])
    builder.add_edge("summarize_file", "reduce_summaries")
    builder.add_edge("reduce_summaries", END)

    return builder.compile()

repo_analyzer_graph = build_repo_analyzer_graph()
print("Repo Analyzer Graph compiled successfully.")


## 9. Subgraph-Inside-Node Adapter Pattern & Streaming Runner
This cell defines:
1. `run_repo_analyzer_with_events(repo_url, query)`: The event streaming execution wrapper.
2. `repo_analyzer_adapter_node(parent_state)`: The **Subgraph-Inside-Node** adapter ensuring parent graphs (DocGen in Phase 6 / Supervisor in Phase 7) remain isolated from internal file trees and reducers.


In [ ]:
async def run_repo_analyzer_with_events(repo_url: str, user_query: str) -> dict:
    """
    Executes the Repo Analyzer with real-time event updates and token streaming.
    Provides complete visibility into what happens inside the workflow.
    """
    start_time = time.time()
    print("=" * 70)
    print("🚀 [WORKFLOW START] Initializing Phase 5 GitHub Repo Analyzer")
    print(f"   Repository: {repo_url}")
    print(f"   Query:      {user_query}")
    print("=" * 70)

    initial_state = {
        "repo_url": repo_url,
        "user_query": user_query,
        "owner": "",
        "repo": "",
        "branch": None,
        "file_tree": [],
        "selected_files": [],
        "selection_reasoning": "",
        "file_summaries": [],
        "combined_summary": "",
        "error": None
    }

    final_state = dict(initial_state)

    # Stream graph updates step-by-step
    async for update in repo_analyzer_graph.astream(initial_state, stream_mode="updates"):
        for node_name, node_output in update.items():
            # Merge updates into accumulated final state
            if isinstance(node_output, dict):
                for k, v in node_output.items():
                    if k == "file_summaries":
                        final_state[k] = final_state.get(k, []) + v
                    else:
                        final_state[k] = v

    elapsed = time.time() - start_time
    print("=" * 70)
    print(f"🏁 [WORKFLOW COMPLETE] Finished in {elapsed:.2f}s")
    print(f"   Total files analyzed: {len(final_state.get('file_summaries', []))}")
    print("=" * 70)
    return final_state


class ExampleParentState(TypedDict):
    """Example parent state for DocGen or Supervisor."""
    query: str
    repo_url: str
    repo_summary: Optional[str]
    analyzed_files: Optional[List[str]]


async def repo_analyzer_adapter_node(state: ExampleParentState) -> dict:
    """
    Isolated adapter node: Parent calls this node without carrying internal
    file trees or operator.add reducers.
    """
    sub_res = await run_repo_analyzer_with_events(state["repo_url"], state["query"])
    return {
        "repo_summary": sub_res.get("combined_summary", ""),
        "analyzed_files": [s["file_path"] for s in sub_res.get("file_summaries", [])]
    }

print("Streaming runner and adapter node ready.")


## 10. Interactive Execution & Demonstration
Run the cell below to see the agent execute with **live event logging and token streaming** on a real GitHub repository!


In [ ]:
# Interactive Test: Real GitHub Repository Inspection with Live Event Streaming
test_repo = "https://github.com/campusx-official/fastapi-demo-api.git"
test_query = "How are the REST API endpoints structured and how does model prediction work in this application?"

# In Jupyter, top-level await is directly supported:
result = await run_repo_analyzer_with_events(test_repo, test_query)

